In [1]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from torch.nn.functional import cosine_similarity
from sklearn.decomposition import PCA

In [2]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
FIG_DIR = "figures"
INPUT_FILE = "text.txt"

os.makedirs(FIG_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# Load model & tokenizer
# ============================================================
login()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="auto",
    output_hidden_states=True
)

model.eval()

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [4]:
INPUT_FILE = "text.txt"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    text = f.read().strip()

max_len = 256
inputs = tokenizer(text, return_tensors="pt",truncation=True,max_length=max_len).to(device)

In [5]:
outputs = model(**inputs, output_hidden_states=True)
hidden_states = outputs.hidden_states

In [18]:
svd_vals = {}      # valeurs singulières
svd_vecs = {}      # vecteurs singuliers droits
k_svd = {}         # dimension retenue
cosine_svd = {}    # cosine similarity après projection
p = 0.999


In [19]:
for layer_id, H in enumerate(hidden_states):

    # -------------------------
    # Données
    # -------------------------
    X = H.squeeze(0).detach().cpu().numpy()  # (T, d)

    # -------------------------
    # SVD non centrée
    # -------------------------
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    V = Vt.T

    svd_vals[layer_id] = S
    svd_vecs[layer_id] = V

    # -------------------------
    # Sélection automatique de k
    # -------------------------
    energy = S**2
    cum_energy = np.cumsum(energy) / np.sum(energy)
    k = np.searchsorted(cum_energy, p) + 1
    k_svd[layer_id] = k

    # -------------------------
    # Projection
    # -------------------------
    X_proj = X @ V[:, :k]

    # -------------------------
    # Cosine similarity projetée
    # -------------------------
    norms = np.linalg.norm(X_proj, axis=1, keepdims=True)
    X_proj_norm = X_proj / (norms + 1e-8)

    cosine_svd[layer_id] = X_proj_norm @ X_proj_norm.T


In [20]:
#Figures — Valeurs singulières
SVD_DIR = os.path.join(FIG_DIR, "svd_vals")
os.makedirs(SVD_DIR, exist_ok=True)

for layer_id, S in svd_vals.items():
    plt.figure(figsize=(6,4))
    plt.plot(S, marker='o', linestyle='-', markersize=3)
    plt.yscale("log")
    plt.xlabel("Singular value index")
    plt.ylabel("Singular value (log scale)")
    plt.title(f"Layer {layer_id} – Singular values")
    plt.grid(True, which="both", ls="--", alpha=0.5)

    fname = os.path.join(SVD_DIR, f"layer_{layer_id}_svd_vals.png")
    plt.savefig(fname, bbox_inches="tight")
    plt.close()


In [ ]:
#Figures — Cosine similarity après projection SVD
COS_SVD_DIR = os.path.join(FIG_DIR, "cosine_svd_proj")
os.makedirs(COS_SVD_DIR, exist_ok=True)

for layer_id, cos_sim in cosine_svd.items():
    plt.figure(figsize=(6,5))
    plt.imshow(cos_sim, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(label="Cosine similarity")
    plt.title(f"Layer {layer_id} – Cosine similarity (SVD p={p})")
    plt.xlabel("Token index")
    plt.ylabel("Token index")
    plt.tight_layout()

    fname = os.path.join(COS_SVD_DIR, f"layer_{layer_id}_cosine_svd_proj.png")
    plt.savefig(fname, bbox_inches="tight")
    plt.close()


In [ ]:
import os
import zipfile
from google.colab import files

# Ensure FIG_DIR is defined
FIG_DIR = "figures"

# Create a zip file
zip_filename = f"{FIG_DIR}.zip"
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for root, dirs, files_in_dir in os.walk(FIG_DIR):
        for file in files_in_dir:
            zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), FIG_DIR))

print(f"Le dossier '{FIG_DIR}' a été compressé dans '{zip_filename}'.")
